# TJK Ganyan Bot - Training Pipeline V4 (FINAL)

Sonuc CSV + Program CSV merge = tam veri (KGS, form, dam_sire, s20 dahil)

Cell'leri sirayla calistir. Full scrape ~25 dk surer, checkpoint'lu.

1. Setup
2. Parser + scraper
3. Hizli test
4. Full scrape (180 gun)
5. Veri kontrolu
6. Feature engineering
7. Feature audit
8. Retrain V2
9. Backtest
10. Sonuclar
11. Model indir

## 1. Setup

In [ ]:
!rm -rf tjk-ganyan-bot
!git clone https://github.com/BerkayMangal/tjk-ganyan-bot.git
%cd tjk-ganyan-bot
!pip install -q requests beautifulsoup4 pandas numpy scikit-learn xgboost lightgbm catboost joblib tqdm
print("Setup OK")

## 2. Parser + Scraper

Sonuc CSV'den: finish position, ganyan, AGF, derece

Program CSV'den: KGS, form (son 6), dam_sire, S20

Ikisini merge ediyoruz.

In [ ]:
import requests, re, csv, io, json, time, os, logging
from datetime import date, datetime, timedelta
from tqdm.auto import tqdm
import pandas as pd
import numpy as np

logging.basicConfig(level=logging.WARNING)

TJK_CDN = "https://medya-cdn.tjk.org/raporftp/TJKPDF"

SESSION = requests.Session()
SESSION.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept-Language': 'tr-TR,tr;q=0.9',
})

HIPODROMLAR = ['Istanbul','Ankara','Izmir','Bursa','Adana','Elazig','Kocaeli','Diyarbakir','Sanliurfa','Antalya']
HIPODROM_TR = {
    'Istanbul':'\u0130stanbul','Ankara':'Ankara','Izmir':'\u0130zmir','Bursa':'Bursa',
    'Adana':'Adana','Elazig':'Elaz\u0131\u011f','Kocaeli':'Kocaeli',
    'Diyarbakir':'Diyarbak\u0131r','Sanliurfa':'\u015eanl\u0131urfa','Antalya':'Antalya',
}
EQUIP_CODES = ['SKG','GKR','KG','DB','SK','OG','DG','AL']


# ============================================================
# CSV FETCH
# ============================================================

def fetch_tjk_csv(target_date, sehir_cdn, csv_type='GunlukYarisSonuclari'):
    y  = target_date.strftime('%Y')
    ymd = target_date.strftime('%Y-%m-%d')
    dmy = target_date.strftime('%d.%m.%Y')
    url = f"{TJK_CDN}/{y}/{ymd}/CSV/{csv_type}/{dmy}-{sehir_cdn}-{csv_type}-TR.csv"
    try:
        resp = SESSION.get(url, timeout=12)
        if resp.status_code != 200:
            return None
        for enc in ['utf-8', 'windows-1254', 'iso-8859-9', 'latin-1']:
            try:
                return resp.content.decode(enc)
            except (UnicodeDecodeError, LookupError):
                continue
    except:
        pass
    return None


# ============================================================
# PARSE HELPERS
# ============================================================

def parse_horse_name(raw_name):
    parts = raw_name.strip().split()
    name_parts = []
    equip_parts = []
    hit_name = False
    for p in reversed(parts):
        if p.upper() in EQUIP_CODES and not hit_name:
            equip_parts.insert(0, p.upper())
        else:
            name_parts.insert(0, p)
            hit_name = True
    if not name_parts and equip_parts:
        name_parts = equip_parts
        equip_parts = []
    return ' '.join(name_parts), ' '.join(equip_parts)


def parse_weight(raw):
    raw = raw.strip()
    m = re.match(r'([\d]+[,.]?\d*)\s*(?:\+\s*([\d]+[,.]?\d*))?', raw)
    if m:
        w = float(m.group(1).replace(',', '.'))
        extra = float(m.group(2).replace(',', '.')) if m.group(2) else 0.0
        return w, extra
    return 57.0, 0.0


def parse_age_text(raw):
    raw = raw.strip()
    am = re.match(r'(\d+)', raw)
    age = int(am.group(1)) if am else 4
    codes = re.findall(r'[adkge]', raw.lower())
    gender = codes[1] if len(codes) >= 2 else ''
    breed_code = codes[-1] if len(codes) >= 3 else ''
    return age, gender, breed_code


def parse_agf(raw):
    m = re.search(r'%(\d+[.,]?\d*)\((\d+)\)', raw)
    if m:
        return float(m.group(1).replace(',', '.')), int(m.group(2))
    return 0.0, 0


def parse_ganyan(raw):
    raw = raw.strip().replace(',', '.')
    m = re.search(r'[\d]+\.?\d*', raw)
    return float(m.group()) if m else 0.0


def parse_race_header(line):
    parts = line.split(';')
    p0 = parts[0] if parts else ''
    rm = re.match(r'(\d+)\.\s*Kos', p0)
    race_number = int(rm.group(1)) if rm else 0
    tm = re.search(r'(\d{1,2}[.:]\d{2})', p0)
    race_time = tm.group(1).replace('.', ':') if tm else ''
    group_name = parts[1].strip() if len(parts) > 1 else ''
    distance = 0
    track_type = ''
    for p in parts:
        dm = re.search(r'(\d{3,4})\s*m', p)
        if dm:
            distance = int(dm.group(1))
        if re.search(r'Kum', p, re.I):
            track_type = 'Kum'
        elif re.search(r'\xc7im|Cim', p, re.I):
            track_type = 'Cim'
        elif re.search(r'Sentetik', p, re.I):
            track_type = 'Sentetik'
    return race_number, race_time, group_name, distance, track_type


# ============================================================
# SONUC CSV PARSER (finish position = satir sirasi)
# ============================================================

def parse_results_csv(text, sehir_name, target_date):
    lines = text.strip().split('\n')
    all_horses = []
    race_number = 0
    race_time = ''
    group_name = ''
    distance = 0
    track_type = ''
    first_prize = 0
    in_horse_data = False
    finish_pos = 0
    race_horses = []

    def flush_race():
        nonlocal race_horses
        if len(race_horses) >= 2:
            n = len(race_horses)
            rid = f"{target_date.strftime('%Y%m%d')}_{sehir_name}_{race_number}"
            for h in race_horses:
                h['race_id'] = rid
                h['n_runners'] = n
            all_horses.extend(race_horses)
        race_horses = []

    for line in lines:
        line = line.strip().lstrip('\ufeff')
        if not line:
            if in_horse_data:
                flush_race()
                in_horse_data = False
            continue

        parts = line.split(';')
        p0 = parts[0].strip()

        if re.match(r'\d+\.\s*Kos', p0):
            if in_horse_data:
                flush_race()
            race_number, race_time, group_name, distance, track_type = parse_race_header(line)
            in_horse_data = False
            finish_pos = 0
            race_horses = []
            continue

        if p0.startswith('Ikramiye') or p0.startswith('\u0130kramiye'):
            continue

        if re.match(r'1\.\)', p0):
            m = re.search(r'1\.\)([\d.]+)', p0)
            if m:
                first_prize = int(m.group(1).replace('.', ''))
            continue

        if p0 == 'At No':
            in_horse_data = True
            finish_pos = 0
            continue

        if p0.startswith('[') or p0.startswith('GANYAN'):
            if in_horse_data:
                flush_race()
                in_horse_data = False
            continue

        if in_horse_data and len(parts) >= 7:
            num_m = re.match(r'(\d+)', p0)
            if not num_m:
                continue

            horse_number = int(num_m.group(1))
            finish_pos += 1

            raw_name = parts[1].strip() if len(parts) > 1 else ''
            horse_name, equipment = parse_horse_name(raw_name)

            age_raw = parts[2].strip() if len(parts) > 2 else '4y'
            age, gender, breed_code = parse_age_text(age_raw)

            sire = parts[3].strip() if len(parts) > 3 else ''
            dam = parts[4].strip() if len(parts) > 4 else ''

            weight_raw = parts[5].strip() if len(parts) > 5 else '57'
            weight, extra_weight = parse_weight(weight_raw)

            jockey = parts[6].strip() if len(parts) > 6 else ''
            jockey = re.sub(r'\s*AP\s*$', '', jockey).strip()

            trainer = parts[8].strip() if len(parts) > 8 else ''

            st_raw = parts[9].strip() if len(parts) > 9 else '0'
            st_m = re.search(r'(\d+)', st_raw)
            start_pos = int(st_m.group(1)) if st_m else horse_number
            is_ds = 'DS' in st_raw.upper()

            agf_raw = parts[10].strip() if len(parts) > 10 else ''
            agf_pct, agf_rank = parse_agf(agf_raw)

            hp_raw = parts[11].strip() if len(parts) > 11 else '0'
            hp_m = re.search(r'(\d+)', hp_raw)
            handicap = int(hp_m.group(1)) if hp_m else 0

            ganyan_raw = parts[13].strip() if len(parts) > 13 else '0'
            ganyan = parse_ganyan(ganyan_raw)

            horse = {
                'race_date': target_date.strftime('%Y-%m-%d'),
                'hippodrome': sehir_name,
                'race_number': race_number,
                'race_time': race_time,
                'group_name': group_name,
                'distance': distance,
                'track_type': track_type,
                'first_prize': first_prize,
                'horse_number': horse_number,
                'horse_name': horse_name,
                'equipment': equipment,
                'age': age,
                'age_text': age_raw,
                'gender': gender,
                'breed_code': breed_code,
                'sire': sire,
                'dam': dam,
                'weight': weight,
                'extra_weight': extra_weight,
                'jockey_name': jockey,
                'trainer_name': trainer,
                'start_position': start_pos,
                'agf_pct': agf_pct,
                'agf_rank': agf_rank,
                'handicap': handicap,
                'ganyan': ganyan,
                'finish_position': finish_pos,
                'is_ds': is_ds,
                'kgs': 0,
                'form': '',
                'dam_sire': '',
                'last_20_score': 0,
            }
            race_horses.append(horse)

    if race_horses:
        flush_race()

    return all_horses


# ============================================================
# PROGRAM CSV PARSER (KGS, form, dam_sire, s20 icin)
# ============================================================

def parse_program_csv(text, sehir_name):
    """
    Program CSV'den ekstra alanlari cek.
    Returns: dict of (race_number, horse_number) -> {kgs, form, dam_sire, last_20_score}
    """
    lines = text.strip().split('\n')
    lookup = {}
    race_number = 0
    in_data = False
    col_map = {}

    for line in lines:
        line = line.strip().lstrip('\ufeff')
        if not line:
            in_data = False
            continue

        parts = line.split(';')
        p0 = parts[0].strip()

        if re.match(r'\d+\.\s*Kos', p0):
            rm = re.match(r'(\d+)', p0)
            race_number = int(rm.group(1)) if rm else 0
            in_data = False
            col_map = {}
            continue

        # Header satiri — kolon index'lerini bul
        if p0 == 'At No':
            in_data = True
            col_map = {}
            for idx, h in enumerate(parts):
                hl = h.strip().lower()
                if hl == 'kgs':
                    col_map['kgs'] = idx
                elif 'son 6' in hl or hl == 'form':
                    col_map['form'] = idx
                elif 's20' in hl or 'son 20' in hl:
                    col_map['s20'] = idx
                elif 'anne baba' in hl or 'annebaba' in hl or 'a.babas' in hl:
                    col_map['dam_sire'] = idx
                elif hl.startswith('orijin') and 'anne' in hl and 'baba' in hl:
                    col_map['dam_sire'] = idx
            continue

        if p0.startswith('Ikramiye') or p0.startswith('\u0130kramiye'):
            continue
        if re.match(r'1\.\)', p0):
            continue
        if p0.startswith('[') or p0.startswith('GANYAN'):
            in_data = False
            continue

        if in_data and len(parts) >= 5:
            num_m = re.match(r'(\d+)', p0)
            if not num_m:
                continue
            hnum = int(num_m.group(1))

            extra = {}

            if 'kgs' in col_map and col_map['kgs'] < len(parts):
                kgs_s = parts[col_map['kgs']].strip()
                km = re.search(r'(\d+)', kgs_s)
                if km:
                    extra['kgs'] = int(km.group(1))

            if 'form' in col_map and col_map['form'] < len(parts):
                form_s = parts[col_map['form']].strip()
                if form_s and form_s != '-':
                    extra['form'] = form_s

            if 's20' in col_map and col_map['s20'] < len(parts):
                s20_s = parts[col_map['s20']].strip()
                sm = re.search(r'(\d+)', s20_s)
                if sm:
                    extra['last_20_score'] = int(sm.group(1))

            if 'dam_sire' in col_map and col_map['dam_sire'] < len(parts):
                ds = parts[col_map['dam_sire']].strip()
                if ds and len(ds) > 1:
                    extra['dam_sire'] = ds

            if extra:
                lookup[(race_number, hnum)] = extra

    return lookup


# ============================================================
# ANA SCRAPE FONKSIYONU (sonuc + program merge)
# ============================================================

def scrape_day(target_date):
    all_horses = []

    for sehir_cdn in HIPODROMLAR:
        sehir_tr = HIPODROM_TR.get(sehir_cdn, sehir_cdn)

        # 1. Sonuc CSV
        text_sonuc = fetch_tjk_csv(target_date, sehir_cdn, 'GunlukYarisSonuclari')
        if not text_sonuc or len(text_sonuc) < 100:
            continue

        horses = parse_results_csv(text_sonuc, sehir_tr, target_date)
        if not horses:
            continue

        # 2. Program CSV — KGS/form/dam_sire/s20 icin
        text_prog = fetch_tjk_csv(target_date, sehir_cdn, 'GunlukYarisProgrami')
        if text_prog and len(text_prog) > 100:
            prog_lookup = parse_program_csv(text_prog, sehir_tr)

            merged = 0
            for h in horses:
                key = (h['race_number'], h['horse_number'])
                extra = prog_lookup.get(key, {})
                if extra:
                    for k, v in extra.items():
                        if v and (k not in h or not h[k] or h[k] == 0 or h[k] == ''):
                            h[k] = v
                    merged += 1

        all_horses.extend(horses)
        time.sleep(0.2)

    return all_horses

print("Parser + Scraper hazir (Sonuc + Program CSV merge)")


## 3. Hizli Test (9 Mart Bursa)

In [ ]:
test_date = date(2026, 3, 9)
test_rows = scrape_day(test_date)
print(f"Tarih: {test_date} -> {len(test_rows)} kayit")

if test_rows:
    df_test = pd.DataFrame(test_rows)
    print(f"Hipodrom: {df_test['hippodrome'].unique().tolist()}")
    print(f"Yaris: {df_test['race_number'].unique().tolist()}")
    print()

    # Ilk yarisin detayi
    r1 = df_test[df_test['race_number'] == 1].head(5)
    for _, h in r1.iterrows():
        print(f"  {h['finish_position']}. {h['horse_name']:20s} | "
              f"J:{h['jockey_name']:12s} | T:{h['trainer_name']:12s} | "
              f"W:{h['weight']}+{h['extra_weight']} | Sire:{h['sire']:12s} | "
              f"AGF:{h['agf_pct']:.1f}%({h['agf_rank']}) | G:{h['ganyan']:.2f} | "
              f"Eq:{h['equipment']} | KGS:{h.get('kgs',0)} | "
              f"Form:{h.get('form','')} | DS:{h.get('dam_sire','')}")
    print()

    # Yeni alanlarin dolulugu
    for col in ['kgs','form','dam_sire','last_20_score']:
        if col in df_test.columns:
            if df_test[col].dtype == 'object':
                filled = (df_test[col].notna() & (df_test[col] != '')).sum()
            else:
                filled = (df_test[col] > 0).sum()
            print(f"  {col:18s}: {filled}/{len(df_test)} dolu")
else:
    print("Veri gelmedi!")


## 4. Full Scrape - Son 180 gun

~25 dk surer. Checkpoint'lu - kesilirse tekrar calistir, devam eder.

`DAYS_BACK = 90` yaparak kisaltabilirsin.

In [ ]:
# Eski veriyi temizle (V3'ten kalan)
for f in ['data/scrape_checkpoint.json', 'data/races_raw.csv', 'data/races_featured.csv']:
    if os.path.exists(f):
        os.remove(f)
        print(f"Silindi: {f}")

DAYS_BACK = 180
RATE_LIMIT = 0.5
OUTPUT_DIR = 'data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

cp_path = os.path.join(OUTPUT_DIR, 'scrape_checkpoint.json')
if os.path.exists(cp_path):
    with open(cp_path) as f:
        cp = json.load(f)
    resume_date = datetime.strptime(cp['last_date'], '%Y-%m-%d').date()
    total_records = cp.get('total_records', 0)
    print(f"Checkpoint: {cp['last_date']} -- {total_records:,} kayit -- devam")
else:
    resume_date = None
    total_records = 0

today = date.today()
start = today - timedelta(days=DAYS_BACK)
if resume_date and resume_date >= start:
    start = resume_date + timedelta(days=1)

n_days = (today - start).days
print(f"{start} -> {today} ({n_days} gun)")
print(f"Tahmini: ~{max(n_days * 0.5 / 60, 1):.0f} dakika")


In [ ]:
csv_path = os.path.join(OUTPUT_DIR, 'races_raw.csv')
csv_exists = os.path.exists(csv_path) and total_records > 0

all_fieldnames = [
    'race_date','hippodrome','race_number','race_time','group_name',
    'distance','track_type','first_prize','horse_number','horse_name',
    'equipment','age','age_text','gender','breed_code','sire','dam','dam_sire',
    'weight','extra_weight','jockey_name','trainer_name','start_position',
    'agf_pct','agf_rank','handicap','ganyan',
    'finish_position','is_ds','kgs','form','last_20_score',
    'race_id','n_runners',
]

csv_file = open(csv_path, 'a' if csv_exists else 'w', newline='', encoding='utf-8')
writer = csv.DictWriter(csv_file, fieldnames=all_fieldnames, extrasaction='ignore')
if not csv_exists:
    writer.writeheader()

day_count = 0
empty_days = 0
current = start
pbar = tqdm(total=(today - start).days, desc="Scraping TJK")

while current <= today:
    try:
        rows = scrape_day(current)
        if rows:
            for row in rows:
                writer.writerow(row)
            total_records += len(rows)
            pbar.set_postfix(date=str(current), rows=len(rows), total=total_records)
        else:
            empty_days += 1

        day_count += 1
        if day_count % 7 == 0:
            csv_file.flush()
            with open(cp_path, 'w') as f:
                json.dump({'last_date': str(current), 'total_records': total_records}, f)

    except Exception as e:
        if day_count < 5:
            print(f"WARN {current}: {e}")

    current += timedelta(days=1)
    pbar.update(1)
    time.sleep(RATE_LIMIT)

pbar.close()
csv_file.close()

with open(cp_path, 'w') as f:
    json.dump({'last_date': str(today), 'total_records': total_records}, f)

print(f"\nScrape tamamlandi!")
print(f"  {day_count} gun | {empty_days} bos gun | {total_records:,} kayit")
print(f"  -> {csv_path}")


## 5. Veri Kontrolu

In [ ]:
df_raw = pd.read_csv(os.path.join(OUTPUT_DIR, 'races_raw.csv'), low_memory=False)
print(f"Toplam: {len(df_raw):,} kayit | {df_raw['race_id'].nunique()} yaris")
print(f"Tarih: {df_raw['race_date'].min()} -> {df_raw['race_date'].max()}")
print(f"Hipodromlar:")
for h, c in df_raw['hippodrome'].value_counts().items():
    print(f"  {h:20s}: {c:>5,}")
print()

has_fp = (df_raw['finish_position'] > 0).sum()
print(f"Sonuclu: {has_fp:,} / {len(df_raw):,} ({has_fp/len(df_raw):.0%})")
print()

print("Kolon doluluk:")
for col in ['horse_name','jockey_name','trainer_name','sire','dam','dam_sire',
            'agf_pct','ganyan','equipment','handicap','kgs','form','last_20_score']:
    if col in df_raw.columns:
        if df_raw[col].dtype == 'object':
            filled = (df_raw[col].notna() & (df_raw[col] != '')).sum()
        else:
            filled = (df_raw[col] > 0).sum()
        print(f"  {col:18s}: {filled:>6,} ({filled/len(df_raw):.0%})")


## 6. Feature Engineering - 82 Feature

In [ ]:
import sys
sys.path.insert(0, '.')
from model.features import FeatureBuilder

df_results = df_raw[
    df_raw['finish_position'].notna() & (df_raw['finish_position'] > 0)
].copy()
print(f"Sonuclu kayit: {len(df_results):,}")

fb = FeatureBuilder()
fb_ok = fb.load()
print(f"FeatureBuilder: {'OK' if fb_ok else 'HATA'} -- {len(fb.feature_cols)} feature")


In [ ]:
feature_rows = []
skipped = 0

for race_id, group in tqdm(df_results.groupby('race_id'), desc="Features"):
    try:
        horses = group.to_dict('records')
        n = len(horses)
        if n < 2:
            skipped += 1
            continue

        row0 = horses[0]
        race_info = {
            'distance': row0.get('distance', 1400),
            'track_type': row0.get('track_type', 'kum'),
            'group_name': row0.get('group_name', ''),
            'hippodrome_name': row0.get('hippodrome', ''),
            'first_prize': row0.get('first_prize', 100000),
            'temperature': 15, 'humidity': 60,
            'race_date': row0.get('race_date'),
        }

        agf_data = []
        for h in horses:
            pct = h.get('agf_pct', 0)
            if not pct or pct <= 0:
                pct = 100.0 / n
            agf_data.append({'horse_number': h.get('horse_number', 0), 'agf_pct': pct, 'is_ekuri': False})
        total_pct = sum(a['agf_pct'] for a in agf_data)
        if total_pct > 0 and abs(total_pct - 100) > 5:
            for a in agf_data:
                a['agf_pct'] = a['agf_pct'] / total_pct * 100

        fb_horses = [{
            'horse_number': h.get('horse_number', 0),
            'horse_name': h.get('horse_name', ''),
            'jockey_name': h.get('jockey_name', ''),
            'trainer_name': h.get('trainer_name', ''),
            'weight': h.get('weight', 57),
            'age': h.get('age', 4),
            'age_text': h.get('age_text', ''),
            'form': h.get('form', ''),
            'equipment': h.get('equipment', ''),
            'kgs': h.get('kgs', 0),
            'last_20_score': h.get('last_20_score', 0),
            'handicap': h.get('handicap', 0),
            'gate_number': h.get('start_position', h.get('horse_number', 1)),
            'sire': h.get('sire', ''),
            'dam': h.get('dam', ''),
            'dam_sire': h.get('dam_sire', ''),
            'total_earnings': 0,
            'extra_weight': h.get('extra_weight', 0),
        } for h in horses]

        matrix, names = fb.build_race_features(fb_horses, race_info, agf_data)

        for i, h in enumerate(horses):
            frow = {
                'race_id': race_id,
                'race_date': h.get('race_date'),
                'hippodrome': h.get('hippodrome', ''),
                'horse_name': h.get('horse_name', ''),
                'horse_number': h.get('horse_number', 0),
                'finish_position': h.get('finish_position', 0),
                'n_runners': n,
            }
            for j, col in enumerate(fb.feature_cols):
                frow[col] = float(matrix[i, j])
            feature_rows.append(frow)

    except Exception as e:
        skipped += 1

print(f"\n{len(feature_rows):,} kayit | {skipped} skip")


In [ ]:
df_feat = pd.DataFrame(feature_rows)
FEAT_CSV = os.path.join(OUTPUT_DIR, 'races_featured.csv')
df_feat.to_csv(FEAT_CSV, index=False)
print(f"{FEAT_CSV} -- {len(df_feat):,} kayit")

fcols = [c for c in df_feat.columns if c.startswith('f_')]
fills = {c: (df_feat[c] != 0).mean() for c in fcols}
avg = np.mean(list(fills.values()))
dead = sum(1 for v in fills.values() if v < 0.01)
print(f"Ortalama fill: {avg:.0%} | Dead: {dead}/{len(fcols)}")

for name, rate in sorted(fills.items(), key=lambda x: x[1])[:10]:
    print(f"  {name:30s} {rate:.1%}")


## 7. Feature Audit

In [ ]:
!python train/feature_audit.py --data data/races_featured.csv --model-dir model/trained

## 8. Retrain V2

2-pass training, AGF noise, ablated model, temporal split.

In [ ]:
!python train/retrain_v2.py \
  --data data/races_featured.csv \
  --output model/trained \
  --labels exponential \
  --agf-noise 0.05 \
  --test-ratio 0.20

## 9. Backtest

In [ ]:
!python train/backtester.py --data data/races_featured.csv --model-dir model/trained

## 10. Sonuclar

In [ ]:
meta_path = 'model/trained/train_meta_v2.json'
if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    print("=" * 60)
    print("RETRAIN V2 SONUCLARI")
    print("=" * 60)
    print(f"Label: {meta.get('label_method')} | AGF noise: {meta.get('agf_noise')}")
    print(f"Train: {meta.get('train_records',0):,} | Test: {meta.get('test_records',0):,}")
    print()
    for name, ev in meta.get('eval', {}).items():
        if ev:
            print(f"  {name:12s} | Top1={ev.get('top1_accuracy',0):.1%} | "
                  f"Top3={ev.get('top3_accuracy',0):.1%} | "
                  f"NDCG@1={ev.get('ndcg1',0):.3f}")
    print()
    for name, imp in meta.get('agf_importance', {}).items():
        if imp is not None:
            s = "KOTU" if imp > 0.5 else "ORTA" if imp > 0.35 else "IYI"
            print(f"  AGF ({name}): {imp:.1%} [{s}]")

bt_path = 'model/trained/backtest_report.json'
if os.path.exists(bt_path):
    with open(bt_path) as f:
        bt = json.load(f)
    print()
    print("=" * 60)
    print("BACKTEST")
    print("=" * 60)
    print(f"  DAR ayak:    {bt.get('dar_leg_hit_rate',0):.1%}")
    print(f"  GENIS ayak:  {bt.get('genis_leg_hit_rate',0):.1%}")
    print(f"  DAR kupon:   {bt.get('dar_kupon_hit_rate',0):.1%}")
    print(f"  GENIS kupon: {bt.get('genis_kupon_hit_rate',0):.1%}")
    print(f"  Top-1:       {bt.get('winner_top1_rate',0):.1%}")
    print(f"  Top-3:       {bt.get('winner_top3_rate',0):.1%}")


## 11. Model Indir

In [ ]:
import zipfile
from google.colab import files

model_files = [
    'model/trained/xgb_ranker.pkl', 'model/trained/lgbm_ranker.pkl',
    'model/trained/cb_ranker.pkl', 'model/trained/ablated_ranker.pkl',
    'model/trained/ablated_columns.json', 'model/trained/scaler.pkl',
    'model/trained/feature_columns.json', 'model/trained/rstats.json',
    'model/trained/train_meta_v2.json', 'model/trained/backtest_report.json',
    'model/trained/feature_audit.json',
]

zip_path = 'data/trained_model_v2.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in model_files:
        if os.path.exists(fp):
            zf.write(fp, os.path.basename(fp))
            print(f"  OK  {os.path.basename(fp)}")
        else:
            print(f"  YOK {os.path.basename(fp)}")

# CSV'leri de ekle (tekrar scrape etmemek icin)
for csv_f in ['data/races_raw.csv', 'data/races_featured.csv']:
    if os.path.exists(csv_f):
        zf_name = os.path.basename(csv_f)
        with zipfile.ZipFile(zip_path, 'a', zipfile.ZIP_DEFLATED) as zf:
            zf.write(csv_f, zf_name)
        print(f"  OK  {zf_name}")

print(f"\n{zip_path}")
files.download(zip_path)


## 12. Railway Push (opsiyonel)

Zip indirdikten sonra:
```bash
unzip trained_model_v2.zip -d model/trained/
git add model/trained/
git commit -m "retrain v2: program csv merge, agf-breaking"
git push
```